# 7. Conclusion and Discussion

This notebook does not train a model, create features, or add figures.

Every statement below is taken from results that were already produced in notebooks 01–06: printed tables, saved metrics, and the written analysis attached to those outputs.

Comparison experiments that are not recorded as executed results in the current notebooks are not treated as findings.


## 7.1 Project Overview

The project predicts daily Rossmann store sales from the supplied historical tables.

The completed workflow is:

Data Understanding
→ Data Cleaning
→ Exploratory Analysis
→ Feature Engineering
→ Model Development
→ Model Evaluation
→ Conclusion

**Data understanding (`01_check.ipynb`).** The raw training table has 1,017,209 daily store rows from 2013-01-01 to 2015-07-31 and includes `Sales` and `Customers`. `store.csv` has 1,115 stores. The official test table has 41,088 rows from 2015-08-01 to 2015-09-17. It has no `Sales` column and no `Customers` column. That absence is part of the original test schema, not a field removed during cleaning.

**Data cleaning (`02_clean.ipynb`).** Cleaning copies were created from the raw files. `StateHoliday` was normalized, `Date` was converted to datetime, and structural missingness in Promo2 and competition fields was preserved rather than filled. The cleaned exports are `data/processed/train_clean.csv` and `data/processed/store_clean.csv`. The raw test file was not rewritten during cleaning.

**Exploratory analysis (`03_eda.ipynb`).** Pattern analysis used operating days (`Open = 1`), because closed stores have zero sales by construction. Operating-day sales are right-skewed: mean 6,955.51 and median 6,369. Extreme operating-day values were retained. Weekday, promotion, holiday, store type, assortment, and competition distance were examined before feature construction.

**Feature engineering (`04_features.ipynb`).** Calendar, store, promotion, and historical demand features were built. Same-day `Customers` was kept out of the modeling columns because it is not available in the test period. Historical customer lags and rolling features were retained. Rolling features were generated on the combined Store–Date timeline and then split back, so the training and test feature sets match. The exported tables are `train_feature_final.csv` (1,017,209 × 40, including `Date` and `Sales`) and `test_feature_final.csv` (41,088 × 39, no `Sales`).

**Model development (`05_model.ipynb`).** The exported tables are loaded independently. The split is time-based: training through 2015-06-30, validation from 2015-07-01 to 2015-07-31, and the official test window from 2015-08-01 to 2015-09-17. Notebook 06 rebuilds that validation window as 34,565 rows. Scaling is fit on the training period only.

**Model evaluation (`06_evaluation.ipynb`).** Evaluation reloads the feature table and the saved artifacts. It does not refit the preprocessor. Validation is rebuilt with the same July 2015 window. The saved network expects 49 encoded features, which matches the transformed validation matrix `(34565, 49)`.

**Conclusion.** This notebook records only those completed steps and their measured results.


## 7.2 Key Findings

### 1. Feature engineering

The exploratory results that motivated the features were measured, not assumed.

On operating days, `Customers` and `Sales` have Pearson correlation 0.8236. Promo days have higher mean sales (8,228.28, n = 376,896) than non-promo days (5,929.41, n = 467,496). `CompetitionDistance` has only a weak linear association with sales (Pearson −0.0364). Weekday differences are visible in the day-of-week sales plot.

The feature-analysis outputs then quantified which constructed columns carry predictive information:

- `Open` has the largest Random Forest importance (0.472434).
- Historical sales features follow: `Sales_lag_14` (0.287071), `Sales_lag_28` (0.062825), `Sales_lag_1` (0.049655), and `Sales_rolling_30` (0.032172).
- `Promo` importance is 0.023480.

Grouped importance, from the executed group table, is:

- Historical demand: 89.35%
- Temporal features: 5.19%
- Promotion features: 4.67%
- Store characteristics: 0.78%

The correlation table agrees on the direction of the main associations: `Sales_lag_14` r = 0.795, `Sales_lag_28` r = 0.779, `Sales_lag_7` r = 0.675, `Customers_lag_7` r = 0.680, `DayOfWeek` r = −0.462, `Promo` r = 0.452, and `Promo2` r = −0.091.

Same-day `Customers` was therefore not used as a model input, even though its raw correlation with sales is strong. Historical customer features were kept because they can be computed from earlier dates. The train–test distribution comparison did not show a severe covariate shift for the compared promotion, weekday, store-type, and historical-sales features.

The Random Forest was used only to interpret features. It is not the forecasting model evaluated in notebook 06.

### 2. Model development

Notebook 05 prepares a time-based split, fits scaling on the training period, and contains an `MLPRegressor` training cell. The artifact actually evaluated in notebook 06 is the saved file `outputs/models/final_sales_model.pkl`.

That saved model, as loaded in notebook 06, is an `MLPRegressor` with hidden layers `(128, 64, 32)`. The transformed validation matrix has 49 features, matching `n_features_in_`.

Notebook 05 records this same architecture and checks the saved file. It does not refit the network, and it does not overwrite the pickle. The evaluation results below belong to that saved model.

Exploration charts for a baseline network, a log-target variant, an architecture comparison, and an early-stopping variant are kept in `outputs/figures/05_model/`. Their bar heights are not printed as exact metrics in notebook 05, so they are not transcribed here. The retained model is the saved 128 → 64 → 32 network, and its printed validation metrics are those in notebook 06.

The model reported as final is therefore the saved network evaluated on the July 2015 validation window. The notebooks do not contain a comparison table that shows it outperforming a documented baseline.

### 3. Model evaluation

On the July 2015 validation set, after setting predictions to zero when `Open = 0`, notebook 06 reports:

- MAE: 491.243061
- RMSE: 731.555819
- R²: 0.958850

The actual-versus-predicted scatter is saved as `outputs/figures/06_evaluation/06_final_actual_vs_predicted.png`. Residual is defined as actual minus predicted. The residual histogram is saved as `outputs/figures/06_evaluation/06_final_residual_distribution.png`. Printed residual summaries are:

- Mean residual: 27.449545
- Residual standard deviation: 731.040655

The mean residual is positive, so the model underpredicts slightly on average. The residual standard deviation is close to the RMSE, so most of the error is spread rather than a large constant shift. The evaluation text that residuals are generally centered is consistent with a mean of about 27 relative to a residual standard deviation of about 731. It does not mean the errors are small.

Error by sales level, using three quantile groups, is:

- Low Sales MAE: 247.110986
- Medium Sales MAE: 481.630176
- High Sales MAE: 745.055776

Absolute error increases with the sales level. High-sales days are the hardest of the three groups in this validation window. The bar chart for this split is displayed in notebook 06 and was not saved as a separate file.


## 7.3 Practical Implications

The measured use of this model is daily store-level sales estimation on a future calendar window, using features that can be known before that day.

On the July 2015 validation window, typical absolute error is about 491 sales units (MAE). Operating-day mean sales in the exploratory analysis were 6,955.51, so this is a useful scale check, not a guarantee. RMSE (731.56) is larger than MAE, which matches the residual spread and the higher error on high-sales days.

The results support using the forecast as one input to planning, for example:

- reviewing which store-days are likely to be busier or quieter;
- comparing a planned promotion period with the model's sales level, because promotion was associated with higher sales in both EDA and the feature analysis;
- flagging high-sales days for extra review, because those days have the largest absolute error.

The model does not replace a manager's decision. It does not guarantee an accurate forecast. Closed-store days are forced to zero after prediction when `Open` is already known to be 0; that rule does not estimate demand for a store that might open. The official August–September 2015 test file has no `Sales`, so the published metrics are validation metrics, not a score on that unlabeled test window.


## 7.4 Limitations and Future Improvements

These limits follow from the current data and the current notebooks. They are not claims about experiments that were not run.

**Same-day customers cannot be used at test time.** The test file has no `Customers` column. The strong operating-day correlation (0.8236) therefore cannot be used as a same-day input. Historical customer lags remain, but they are not a substitute for same-day traffic.

**The feature set is the Rossmann table plus engineered columns.** Weather, local events, and prices were not added and were not tested. `CompetitionDistance` is included, but its linear correlation with sales is weak (−0.0364), so it should not be described as a strong external driver in this project.

**The network is not a sequence model.** Lags and rolling statistics are static columns passed to an `MLPRegressor`. The model does not consume an ordered sequence directly.

**Validation is one future month inside the labeled history.** Metrics are computed on 1–31 July 2015. The official test period has no labels in this project, so generalization to August–September 2015 was not measured.

**Later test dates often lack complete rolling windows.** Test-period `Sales` and `Customers` stay missing, and rolling features look backward. Training rolling values are unaffected, but many test rows do not have a full recent window.

**The saved weights were not retrained in this review.** Notebook 05 now records the saved 128 → 64 → 32 architecture and does not overwrite the existing pickle. The printed validation metrics still come from notebook 06.

**Exploration charts do not have printed bar values.** Files under `outputs/figures/05_model/` compare baseline, log-target, architecture, and early-stopping variants. Those charts are retained, but their exact bar heights are not printed in the notebook and are not treated as official metrics.

Possible later work, not implemented in this project:

- add external features such as weather or local events, then re-evaluate leakage and test availability;
- compare the saved network with a true sequence or time-series model on the same July 2015 split;
- keep the evaluated `(128, 64, 32)` artifact separate from any new training cell so validation metrics remain traceable.
